# 09 - RQ4: EPSS alone vs text alone vs text + EPSS, same test rows
Fusion fitted on val_a, calibration and Conformal on val_b, test once.

In [ ]:
# Colab setup: repository at /content/kev-text (git clone or unzip), then run all.
import os, sys
REPO = "/content/kev-text"
if os.path.exists(REPO):
    os.chdir(REPO)
sys.path.insert(0, os.getcwd())
# !pip install -q -r requirements.txt          # once per Colab session
import numpy as np, pandas as pd
from src import config, io_utils, labels, evaluate
SYN = False   # True = dry run on src.synthetic; everything goes to outputs_synthetic/

In [ ]:
if SYN:
    from src import synthetic
    df = synthetic.make(6000)
    if not config.SPLITS_LOCK.exists():                   # mirrors the extras notebook 02 writes
        from src import sensitivity
        sp0 = labels.assign_split(df["published_date"])
        m = sensitivity.kev_retroactive_monthly(df[["cve_id", "kev_date_added"]].dropna(),
                                                df[["cve_id", "published_date"]])
        vb = df[(sp0 == "val") & (df["published_date"] > config.VAL_A_END)]
        labels.write_lock(df, "initial_seal", "synthetic dry run",
                          extra={"s1": sensitivity.s1_record(m, df, sp0),
                                 "conformal_alpha": config.conformal_alpha(int(vb["label_12m"].sum()))})
else:
    df = pd.read_parquet(config.DATA / "dataset_labelled.parquet")
lock = labels.require_lock()
X, y, split = labels.get_xy(df, labels.CLEAN)
split.value_counts()

In [ ]:
from src import fusion, plots
TEXT_MODEL = io_utils.read_json(config.SELECTION_LOCK).get("best_encoder") or "tfidf-lr"
s = {sp: evaluate.load_scores(TEXT_MODEL, sp, is_synthetic=SYN) for sp in ("val_a", "val_b", "test")}
test_idx, counts = fusion.comparison_rows(df, s["test"].index)
yt = y[test_idx]
counts

In [ ]:
out, rel = {}, {}
for variant in ("epss", "text", "text+epss", "text+epss+cvss+cwe"):
    fz = fusion.Fusion(variant).fit(s["val_a"], y[s["val_a"].index], df)
    cal = fusion.Calibrator("isotonic").fit(fz.predict(s["val_b"], df), y[s["val_b"].index])
    out[variant] = pd.Series(cal.transform(fz.predict(s["test"].loc[test_idx], df)), index=test_idx)
    rel[variant] = evaluate.reliability_table(yt, out[variant])
table = pd.DataFrame({v: evaluate.metrics(yt, p) for v, p in out.items()}).T
primary = evaluate.paired_bootstrap(yt, out["text+epss"], out["epss"])    # the pre-registered test
evaluate.save_table(table, "table_rq4", is_synthetic=SYN)
table, primary

In [ ]:
ver = df.loc[test_idx, "epss_model_version"]
pd.DataFrame({v: evaluate.metrics(yt[ver == v], out["text+epss"][ver == v]) for v in ver.dropna().unique()}).T

In [ ]:
fz = fusion.Fusion("text+epss").fit(s["val_a"], y[s["val_a"].index], df)
mc = fusion.MondrianConformal().fit(fz.predict(s["val_b"], df), y[s["val_b"].index],
                                    {int(k): v for k, v in lock["conformal_alpha"].items()})
cov_q = {str(q): mc.coverage(fz.predict(s["test"].loc[idx], df), y[idx])
         for q, idx in pd.Series(test_idx, index=test_idx).groupby(df.loc[test_idx, "published_date"].dt.to_period("Q"))}
pd.DataFrame(cov_q).T                     # coverage per class and per quarter

In [ ]:
plots.save_figure(plots.reliability_diagram(rel), "fig_calibration", None, SYN)